<a href="https://colab.research.google.com/github/lionzionion/-Enhanced-Calculator-Project-/blob/main/Explainer_IA_Ditto_UM_PLAY_CORRIGIDO_ONNX_1202.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 Explainer IA — Ditto Realista (correção ONNX 1.20.2)

**Aperte Play na próxima célula.** O notebook pede foto e frase, gera a voz em português e exporta vídeo vertical MP4.

### Correção desta versão
O ambiente Python 3.10 usava `onnxruntime-gpu==1.20.1`, que não foi encontrado pelo pip. Agora instala `onnxruntime-gpu==1.20.2`, versão com pacote Linux CPython 3.10. Os demais passos foram mantidos.

**Observação:** a primeira execução requer downloads grandes do PyTorch e dos modelos Ditto (aproximadamente 2,3 GB em checkpoints). Se usar o mesmo ambiente de execução, as bibliotecas já baixadas serão reaproveitadas pelo instalador. A saída do Ditto na GPU T4 ainda depende de validação real no Colab.


In [5]:
# @title ▶️ GERAR VÍDEO REALISTA — foto + frase + Ditto + download
# Um único Play: recebe entrada, instala motor na primeira sessão e cria MP4.
from pathlib import Path
from io import BytesIO
from datetime import datetime
from collections import deque
from IPython.display import display, HTML, Video
from google.colab import files
from PIL import Image, ImageOps
import subprocess, os, sys, re, time, unicodedata, shutil, importlib.util

# CONFIGURAÇÕES PADRÃO (não precisa editar)
VOZ = 'pt-BR-AntonioNeural'
PITCH = '-10Hz'
RATE = '-3%'
OUTPUT_W, OUTPUT_H = 1080, 1920
ROOT = Path('/content')
ENGINE = ROOT / 'ditto-talkinghead'
CHECKPOINTS = ENGINE / 'checkpoints'
PY_ENV = ROOT / 'explainer_ditto_py310'
PY310 = PY_ENV / 'bin' / 'python'
MINIFORGE = ROOT / 'explainer_miniforge'  # Reaproveita a instalação do notebook anterior
CONDA = MINIFORGE / 'bin' / 'conda'
LOG = ROOT / 'explainer_ditto_log.txt'
OUTDIR = ROOT / 'explainer_ditto_resultados'
OUTDIR.mkdir(exist_ok=True, parents=True)


def run(args, cwd=None, description=None):
    """Executa comando e sempre preserva as últimas linhas do erro."""
    args = [str(x) for x in args]
    if description:
        print('\n🔧 ' + description, flush=True)
    tail = deque(maxlen=100)
    last_print = 0

    # Copia as variáveis de ambiente atuais e limpa backend específico do Colab/Matplotlib
    # para evitar conflitos de ValueError no backend dentro do Conda.
    custom_env = os.environ.copy()
    custom_env.pop('MPLBACKEND', None)

    with LOG.open('a', encoding='utf-8') as logfile:
        logfile.write('\n$ ' + ' '.join(args) + '\n')
        logfile.flush()
        p = subprocess.Popen(args, cwd=str(cwd) if cwd else None,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             env=custom_env,
                             text=True, errors='replace', bufsize=1)
        for line in p.stdout:
            logfile.write(line)
            logfile.flush()
            tail.append(line.rstrip())
            now = time.monotonic()
            if now - last_print > 8:
                print('    ⏳ ' + line.strip()[:200], flush=True)
                last_print = now
        rc = p.wait()
    if rc:
        print('\n❌ ETAPA COM ERRO:', description or args[0])
        print('--- ÚLTIMAS LINHAS DO ERRO ---')
        print('\n'.join(tail)[-14000:])
        print('--- FIM DO ERRO ---')
        print('📄 Log:', LOG)
        raise RuntimeError(f'Falha no comando (código {rc}). Consulte o erro acima.')


def safe_name(text):
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    return re.sub(r'[^a-z0-9]+', '_', text.lower()).strip('_')[:42] or 'teste'


# ENTRADAS: SÃO PEDIDAS NO MOMENTO DO PLAY
print('📸 PASSO 1/5 — Selecione uma FOTO nítida do apresentador:')
items = files.upload()
images = [(name, data) for name, data in items.items()
          if name.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
if not images:
    raise ValueError('Envie uma imagem JPG, JPEG, PNG ou WEBP e clique novamente no Play.')
filename, contents = images[0]
ID = safe_name(filename.rsplit('.', 1)[0]) + '_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR = OUTDIR / ID
RUN_DIR.mkdir(exist_ok=False)
PHOTO = RUN_DIR / 'foto.jpg'
with Image.open(BytesIO(contents)) as src:
    src = ImageOps.exif_transpose(src).convert('RGB')
    src.save(PHOTO, 'JPEG', quality=96, subsampling=0)
    if min(src.size) < 700:
        print('⚠️ Foto com resolução baixa: prefira imagem nítida de pelo menos 1024 px.')
print('✅ Foto recebida:', filename)

print('\n📝 PASSO 2/5 — Digite a frase a ser narrada e pressione Enter:')
TEXT = input('Frase do áudio: ').strip()
if not TEXT:
    raise ValueError('A frase não pode estar vazia.')
if len(TEXT) > 1800:
    raise ValueError('Faça o teste com até 1.800 caracteres por vídeo.')
print('✅ Frase recebida. Preparando motor Ditto...')

if shutil.which('nvidia-smi') is None:
    raise RuntimeError('GPU NVIDIA não detectada. No Colab, altere o ambiente de execução para GPU T4 ou superior.')
run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], description='Identificando GPU NVIDIA')
if shutil.which('ffmpeg') is None:
    run(['apt-get', 'update', '-qq'], description='Preparando sistema')
    run(['apt-get', 'install', '-y', 'ffmpeg'], description='Instalando FFmpeg')

print('\n⚙️ PASSO 3/5 — Instalando/verificando Ditto (versão PyTorch, sem TensorRT)...')
# Clone só na primeira execução; não baixa SadTalker.
if not (ENGINE / 'inference.py').is_file():
    if ENGINE.exists():
        shutil.rmtree(ENGINE)
    run(['git', 'clone', '--depth', '1',
         'https://github.com/antgroup/ditto-talkinghead.git', ENGINE],
        description='Baixando código oficial do Ditto')

# Python 3.10 isolado: funciona mesmo quando o Colab usa Python 3.13.
if not CONDA.exists():
    installer = ROOT / 'ditto_miniforge.sh'
    if MINIFORGE.exists():
        shutil.rmtree(MINIFORGE)
    run(['curl', '-fL', '--retry', '3', '--retry-delay', '3',
         'https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh',
         '-o', installer], description='Obtendo gerenciador Python 3.10')
    run(['bash', installer, '-b', '-p', MINIFORGE],
        description='Instalando gerenciador de ambientes')
    installer.unlink(missing_ok=True)

if not PY310.is_file():
    run([CONDA, 'create', '--yes', '--prefix', PY_ENV,
         '--override-channels', '-c', 'conda-forge', 'python=3.10', 'pip'],
        description='Criando Python 3.10 isolado')

READY = PY_ENV / '.ditto_pytorch_v4_einops'
if not READY.is_file():
    run([PY310, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'],
        description='Preparando pip')
    # CUDA 12.1: PyTorch 2.5.1 é a base usada pelo ambiente oficial do Ditto.
    run([PY310, '-m', 'pip', 'install', '--progress-bar', 'off',
         '--index-url', 'https://download.pytorch.org/whl/cu121',
         'torch==2.5.1', 'torchvision==0.20.1', 'torchaudio==2.5.1'],
        description='Instalando PyTorch CUDA')
    # Somente PyTorch, sem TensorRT. Os detectores auxiliares ONNX são executados
    # pelo onnxruntime-gpu com fallback para CPU se o provedor CUDA não carregar.
    run([PY310, '-m', 'pip', 'install', '--progress-bar', 'off',
         'numpy==2.0.1', 'scipy==1.15.0', 'librosa==0.10.2.post1',
         'numba==0.60.0', 'opencv-python-headless==4.10.0.84',
         'scikit-image==0.25.0', 'imageio==2.36.1', 'imageio-ffmpeg==0.5.1',
         'filetype==1.2.0', 'cython==3.0.11', 'onnxruntime-gpu==1.20.2',
         'colored', 'tqdm==4.67.1', 'PyYAML==6.0.2', 'huggingface_hub==0.34.4',
         'mediapipe', 'einops'],
        description='Instalando dependências de vídeo, face e áudio')
    run([PY310, '-c',
         'import torch, cv2, librosa, onnxruntime, numpy, mediapipe, einops; '
         'print("PYTORCH",torch.__version__,"CUDA",torch.cuda.is_available()); '
         'assert torch.cuda.is_available(), "CUDA indisponível"; '
         'print("GPU",torch.cuda.get_device_name(0)); '
         'print("ONNX",onnxruntime.__version__,onnxruntime.get_available_providers()); '],
        description='Validando biblioteca e GPU')

    # Verifica se a versão disponível no PyPI foi instalada corretamente.
    run([PY310, '-c', 'import onnxruntime as ort; assert ort.__version__ == "1.20.2", ort.__version__; print("ONNX Runtime GPU confirmado:", ort.__version__)'],
        description='Confirmando versão ONNX Runtime 1.20.2')
    READY.touch()
else:
    print('✅ Ambiente Ditto reutilizado desta sessão.')

# Baixa apenas modelos PyTorch necessários; evita os motores TensorRT e ONNX gigantes.
MODEL_FILES = [
    'ditto_cfg/v0.4_hubert_cfg_pytorch.pkl',
    'ditto_pytorch/aux_models/2d106det.onnx',
    'ditto_pytorch/aux_models/det_10g.onnx',
    'ditto_pytorch/aux_models/face_landmarker.task',
    'ditto_pytorch/aux_models/hubert_streaming_fix_kv.onnx',
    'ditto_pytorch/aux_models/landmark203.onnx',
    'ditto_pytorch/models/appearance_extractor.pth',
    'ditto_pytorch/models/decoder.pth',
    'ditto_pytorch/models/lmdm_v0.4_hubert.pth',
    'ditto_pytorch/models/motion_extractor.pth',
    'ditto_pytorch/models/stitch_network.pth',
    'ditto_pytorch/models/warp_network.pth',
]
if not all((CHECKPOINTS / relative).is_file() and (CHECKPOINTS / relative).stat().st_size > 500
           for relative in MODEL_FILES):
    # Executa snapshot_download em Python 3.10, sem exigir o pacote no kernel Python 3.13.
    download_py = RUN_DIR / 'baixar_modelos.py'
    download_py.write_text(
        "from huggingface_hub import snapshot_download\n"
        + "snapshot_download(repo_id='digital-avatar/ditto-talkinghead', "
        + "repo_type='model', "
        + "allow_patterns=['ditto_cfg/v0.4_hubert_cfg_pytorch.pkl', 'ditto_pytorch/**'], "
        + f"local_dir={str(CHECKPOINTS)!r}, max_workers=4)\n",
        encoding='utf-8'
    )
    run([PY310, download_py], description='Baixando os modelos oficiais PyTorch do Ditto')
missing = [x for x in MODEL_FILES if not (CHECKPOINTS / x).is_file()]
if missing:
    raise RuntimeError('Alguns checkpoints não foram baixados: ' + ', '.join(missing) +
                       '. Confira o log e rode novamente para retomar o download.')
print('✅ Pesos do Ditto encontrados.')

print('\n🎙️ PASSO 4/5 — Gerando voz brasileira e animando o rosto...')
# Edge TTS usa conexão com Microsoft, mas não exige chave de API.
if importlib.util.find_spec('edge_tts') is None:
    run([sys.executable, '-m', 'pip', 'install', '-q', 'edge-tts'],
        description='Instalando Edge TTS no Colab')
import edge_tts
MP3 = RUN_DIR / 'narracao.mp3'
WAV = RUN_DIR / 'narracao_16khz.wav'
async def synthesize():
    tts = edge_tts.Communicate(TEXT, VOZ, pitch=PITCH, rate=RATE)
    await tts.save(str(MP3))
await synthesize()
if not MP3.is_file() or MP3.stat().st_size < 1000:
    raise RuntimeError('O Edge TTS não retornou áudio válido.')
run(['ffmpeg', '-hide_banner', '-loglevel', 'error', '-y', '-i', MP3,
     '-ar', '16000', '-ac', '1', '-c:a', 'pcm_s16le', WAV],
    description='Preparando áudio de 16 kHz para o Ditto')

RAW_MP4 = RUN_DIR / 'ditto_original.mp4'
run([PY310, 'inference.py',
     '--data_root', CHECKPOINTS / 'ditto_pytorch',
     '--cfg_pkl', CHECKPOINTS / 'ditto_cfg/v0.4_hubert_cfg_pytorch.pkl',
     '--audio_path', WAV, '--source_path', PHOTO,
     '--output_path', RAW_MP4],
    cwd=ENGINE, description='Renderizando expressões e lábios com Ditto PyTorch')
if not RAW_MP4.is_file() or RAW_MP4.stat().st_size < 10_000:
    raise RuntimeError('O Ditto terminou sem um MP4 válido; confira o log detalhado.')

print('\n📱 PASSO 5/5 — Exportando MP4 vertical 1080×1920...')
FINAL = ROOT / ('explainer_ditto_realista_' + ID + '.mp4')
# Fundo suave desfocado evita cortar o rosto quando a saída do modelo for quadrada.
filter_graph = (
    f'[0:v]split=2[bg][fg];'
    f'[bg]scale={OUTPUT_W}:{OUTPUT_H}:force_original_aspect_ratio=increase,'
    f'crop={OUTPUT_W}:{OUTPUT_H},boxblur=12:1[background];'
    f'[fg]scale={OUTPUT_W}:{OUTPUT_H}:force_original_aspect_ratio=decrease[foreground];'
    f'[background][foreground]overlay=(W-w)/2:(H-h)/2:shortest=1,'
    f'format=yuv420p[v]'
)
run(['ffmpeg', '-hide_banner', '-loglevel', 'error', '-y', '-i', RAW_MP4,
     '-i', WAV, '-filter_complex', filter_graph,
     '-map', '[v]', '-map', '1:a:0', '-shortest', '-r', '25',
     '-c:v', 'libx264', '-crf', '18', '-preset', 'medium',
     '-c:a', 'aac', '-b:a', '192k', '-movflags', '+faststart', FINAL],
    description='Finalizando H.264 + AAC para celular')
if not FINAL.exists() or FINAL.stat().st_size < 10_000:
    raise RuntimeError('A exportação final não produziu um MP4 válido.')
print('\n✅ VÍDEO REALISTA GERADO!')
print('📂', FINAL)
print('📦 %.1f MB' % (FINAL.stat().st_size / (1024 * 1024)))

# DOWNLOAD PRIMEIRO: um erro do player nunca impede a entrega, como na versão anterior.
print('⬇️ Baixando MP4 automaticamente...')
try:
    files.download(str(FINAL))
except Exception as exc:
    print('⚠️ O download automático não abriu:', str(exc))
    print('Abra o painel de arquivos do Colab e baixe este caminho:', FINAL)

# Prévia só após tentar entregar o arquivo.
try:
    if FINAL.stat().st_size < 35 * 1024 * 1024:
        display(Video(data=FINAL.read_bytes(), embed=True, mimetype='video/mp4', width=340))
    else:
        print('Prévia ignorada porque o vídeo é grande. MP4 está salvo e disponível.')
except Exception as exc:
    print('ℹ️ Prévia indisponível, mas o arquivo está gerado:', str(exc))
print('\n🔄 Para outro vídeo, aperte novamente o ▶️ desta célula.')

📸 PASSO 1/5 — Selecione uma FOTO nítida do apresentador:


Saving WhatsApp Image 2026-10-25 at 13.58.17.jpeg to WhatsApp Image 2026-10-25 at 13.58.17.jpeg
✅ Foto recebida: WhatsApp Image 2026-10-25 at 13.58.17.jpeg

📝 PASSO 2/5 — Digite a frase a ser narrada e pressione Enter:
Frase do áudio: Oi, gente! Tudo bem com vocês? Eu sou apaixonada por encontrar produtos incríveis, úteis e com aquele precinho que a gente ama! Aqui eu compartilho achadinhos, ofertas especiais e dicas para facilitar o seu dia a dia. Então já me acompanha, porque vem muita novidade boa por aí!
✅ Frase recebida. Preparando motor Ditto...

🔧 Identificando GPU NVIDIA
    ⏳ Tesla T4, 15360 MiB

⚙️ PASSO 3/5 — Instalando/verificando Ditto (versão PyTorch, sem TensorRT)...
✅ Ambiente Ditto reutilizado desta sessão.
✅ Pesos do Ditto encontrados.

🎙️ PASSO 4/5 — Gerando voz brasileira e animando o rosto...

🔧 Preparando áudio de 16 kHz para o Ditto

🔧 Renderizando expressões e lábios com Ditto PyTorch
    ⏳ 2026-10-25 17:01:06.746727927 [E:onnxruntime:Default, provider_bridge_or

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🔄 Para outro vídeo, aperte novamente o ▶️ desta célula.


## Dicas para realismo

- Prefira fotos com **1024 px ou mais**, rosto de frente, olhos e boca visíveis.
- Gere inicialmente frases curtas (10–20 segundos).
- A renderização do Ditto pode exigir mais memória e tempo que o SadTalker. Se a T4 não suportar a sua execução, tente uma GPU mais potente.
- Esta é uma integração nova com o código oficial do modelo; **não foi executada ponta a ponta no Colab**.
- Execute o mesmo Play novamente para criar outros vídeos, enquanto a sessão estiver ativa.
